**IN:** directory of pdfs we want to extract info from, chromadb vector databases for each of those pdfs

**OUT:** csv that follows the same format as the ERA training data

In [33]:
# Import libraries
import chromadb
from openai import OpenAI
from dotenv import load_dotenv
import csv
from datetime import datetime
import re
import os

In [34]:
load_dotenv()  # Loads OPENAI_API_KEY

# Correct way to extract paper IDs
paper_ids = [
    f.split('-')[0]
    for f in os.listdir("pdfs")
    if f.endswith('.pdf')
]

pre_prompt = """ 
You are a helpful assistant. You answer questions about diet information in livestock management scientific literature. 
But you only answer based on knowledge I'm providing you. You don't use your internal knowledge and you don't make things up.
If you don't know the answer, just say: I don't know
"""

user_query = """
Give a csv table for each diet in this paper. 
Each row will have an ingredient in the diet, the ingredient type, 
the amount of the ingredient, the units for that amount, 
whether the ingredient is dry, and whether it was fed ad libitum.
Ingredient types include Crop Byproduct, Crop Product, Forage Plants, Supplement, and Other Ingredients.
Here is an example of a diet csv table: 
Ingredient,Ingredient Type,Amount,Units,Dry,Fed Ad Libitum
Yellow maize meal,Crop Product,22.0,%,Yes,Yes
Eragrostis curvula,Forage Plants,18.0,%,Yes,Yes
Leucaena leucocephala,Forage Plants,25.0,%,Yes,Yes
Wheat bran,Crop Byproduct,8.0,%,Yes,Yes
"""

In [35]:
import os
import chromadb
from chromadb.utils import embedding_functions

CHROMA_PATH = "/Users/pk_3/My_Documents/CTC/Alliance-Bioversity-CIAT/chroma_db"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = chroma_client.get_or_create_collection(
    name="papers",
    embedding_function=embedding_fn
)

outputs = []

# Rebuild the mapping automatically
pdf_files = [f for f in os.listdir("pdfs") if f.endswith('.pdf')]
pdf_map = {f.split('-')[0]: f for f in pdf_files}

for PAPER_ID, pdf_file in pdf_map.items():

    results = collection.query(
        query_texts=[user_query],
        n_results=50,
        where={"paper_id": PAPER_ID}
    )

    client = OpenAI()

    system_prompt = f"""
    {pre_prompt}
    --------------------
    Data retrieved for {PAPER_ID}:
    {results['documents']}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ]
    )

    outputs.append({
        "paper_id": PAPER_ID,
        "pdf_file": pdf_file,
        "user_query": user_query,
        "llm_output": response.choices[0].message.content
    })


In [39]:
log_filename = "llm_log.csv"
csv_filename = "diet_tables.csv"

if not os.path.exists(csv_filename):
    with open(csv_filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "paper_id",
            "pdf_filename",
            "table_number",
            "Ingredient",
            "Ingredient Type",
            "Amount",
            "Units",
            "Dry",
            "Fed Ad Libitum"
        ])

for item in outputs:
    paper_id   = item["paper_id"]
    pdf_file   = item["pdf_file"]
    user_query = item["user_query"]
    output_text = item["llm_output"]

    if not os.path.exists(log_filename):
        with open(log_filename, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["timestamp", "paper_id", "pdf_filename", "user_query", "llm_response"])

    with open(log_filename, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            datetime.now().isoformat(),
            paper_id,
            pdf_file,
            user_query,
            output_text
        ])

    tables = re.findall(r"((?:\|.*?\|\n)+)", output_text)

    if tables:
        table_num = 0

        for table in tables:
            lines = [ln.strip() for ln in table.split("\n") if ln.strip()]
            if len(lines) < 3:
                continue

            cleaned = [re.sub(r"^\||\|$", "", ln) for ln in lines]
            rows = [re.split(r"\s*\|\s*", ln) for ln in cleaned]

            data_rows = rows[2:]
            if not data_rows:
                continue

            table_num += 1

            with open(csv_filename, "a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                for r in data_rows:
                    if len(r) < 6:
                        continue
                    writer.writerow([
                        paper_id,
                        pdf_file,
                        table_num,
                        *[c.strip() for c in r[:6]]
                    ])
    else:
        with open(csv_filename, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([
                paper_id,
                pdf_file,
                "no_table",
                output_text,
                "", "", "", "", ""
            ])
